In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from IPython.display import Image, display

In [ ]:
chat_template_1 = ChatPromptTemplate.from_template('''
What are the five most important skills a {football position} needs?
Answer only by listing the skills.
''')

chat_template_2 = ChatPromptTemplate.from_template('''
Considering the football skills provided, develop a strategy for effectively learning and mastering them:
{skills}
''')

In [ ]:
chat_template_1

In [ ]:
class FootballPositions(BaseModel):
    """Five most important skills for a given position in football."""

    footballskills: list[str] = Field(
        min_length=5,
        max_length=5,
        description="Five most important skills for the position, in priority order",
    )


class LearningStrategy(BaseModel):
    """Strategy for learning and mastering football positions."""

    strategy: str = Field(
        description="A strategy for effectively learning and mastering the skills",
    )

In [ ]:
llm = ChatLiteLLM(
    model="gpt-5.4-mini",
    temperature=0,
    model_kwargs={'seed': 365},
    max_tokens=100,
)

In [ ]:
llm_1 = llm.with_structured_output(FootballPositions)
llm_2 = llm.with_structured_output(LearningStrategy)

In [ ]:
chain_long = (
    chat_template_1 | llm_1 | {'skills': RunnablePassthrough()} |
    chat_template_2 | llm_2
)

In [ ]:
display(Image(chain_long.get_graph().draw_mermaid_png()))